In [ ]:
# ── Naive baseline column ──────────────────────────────────────────────────────

# def _horizon_naive_col(h: int) -> str:
#     """Same-day naive baseline column.

#     Feature data is at 5-min resolution; lag 336 = 28h (closest upstream lag
#     to same-time-yesterday at 24h = 288 steps, which is not in the lag set).
#     """
#     return f"{os.environ['TARGET'].lower()}_price_lag_336"

In [ ]:
# def _spike_policy_score(y_true: np.ndarray, y_pred: np.ndarray, spike_threshold: float) -> float:
#     mae = float(np.mean(np.abs(y_true - y_pred)))
#     spike = y_true > spike_threshold
#     dip = y_true < _DIP_THRESHOLD
#     if int(spike.sum()) == 0 or int((~spike).sum()) == 0:
#         return mae
#     spike_mae = float(np.mean(np.abs(y_true[spike] - y_pred[spike])))
#     non_mae   = float(np.mean(np.abs(y_true[~spike] - y_pred[~spike])))
#     dip_mae   = float(np.mean(np.abs(y_true[dip] - y_pred[dip]))) if int(dip.sum()) > 0 else non_mae
#     return 2.2 * spike_mae + 1.3 * dip_mae + 1.0 * non_mae + 0.25 * mae

In [ ]:
# def _pick_spike_source(
#     y_base: np.ndarray,
#     y_spike: np.ndarray,
#     y_q: np.ndarray,
#     src: int,
# ) -> np.ndarray:
#     if src == 1:
#         return y_q
#     if src == 2:
#         return np.maximum(y_spike, y_q).astype(np.float32)
#     return y_spike


# def _apply_spike_policy(
#     y_base: np.ndarray,
#     spike_prob: np.ndarray,
#     y_spike: np.ndarray,
#     y_q: np.ndarray,
#     kind: int,
#     p1: float,
#     p2: float,
#     p3: float,
#     src: int,
# ) -> np.ndarray:
#     src_pred = _pick_spike_source(y_base, y_spike, y_q, src)
#     if kind == 0:
#         return ((1.0 - spike_prob) * y_base + spike_prob * src_pred).astype(np.float32)
#     if kind == 1:
#         return _spike_blend(y_base, spike_prob, src_pred, p1, p2, p3)
#     gate = spike_prob >= p1
#     uplift = np.maximum(src_pred - y_base, 0.0)
#     out = y_base.copy()
#     out[gate] = y_base[gate] + float(p2) * uplift[gate]
#     return out.astype(np.float32)


In [ ]:
# def _spike_blend(
#     y_base: np.ndarray,
#     spike_prob: np.ndarray,
#     y_spike: np.ndarray,
#     p_thr: float,
#     p_pow: float,
#     w_max: float,
# ) -> np.ndarray:
#     p_adj = np.clip((spike_prob - p_thr) / (1.0 - p_thr + 1e-6), 0.0, 1.0)
#     w     = (np.power(p_adj, p_pow) * w_max).astype(np.float32)
#     delta = np.maximum(y_spike - y_base, 0.0)
#     return (y_base + w * delta).astype(np.float32)


In [ ]:
# def _dip_blend(
#     y_base: np.ndarray,
#     dip_prob: np.ndarray,
#     y_dip: np.ndarray,
#     p_thr: float,
#     p_pow: float,
#     w_max: float,
# ) -> np.ndarray:
#     p_adj = np.clip((dip_prob - p_thr) / (1.0 - p_thr + 1e-6), 0.0, 1.0)
#     w     = (np.power(p_adj, p_pow) * w_max).astype(np.float32)
#     down  = np.maximum(y_base - y_dip, 0.0)
#     return (y_base - w * down).astype(np.float32)


# def _apply_dip_policy(
#     y_base: np.ndarray,
#     dip_prob: np.ndarray,
#     y_dip: np.ndarray,
#     kind: int,
#     p1: float,
#     p2: float,
#     p3: float,
# ) -> np.ndarray:
#     if kind == 0:
#         return ((1.0 - dip_prob) * y_base + dip_prob * y_dip).astype(np.float32)
#     if kind == 1:
#         return _dip_blend(y_base, dip_prob, y_dip, p1, p2, p3)
#     gate = dip_prob >= p1
#     down = np.maximum(y_base - y_dip, 0.0)
#     out = y_base.copy()
#     out[gate] = y_base[gate] - float(p2) * down[gate]
#     return out.astype(np.float32)


In [ ]:
 # def _tune_and_calibrate() -> dict:
    #     """
    #     Tune per-horizon blend weights and spike/dip correction policies on the
    #     validation set, then fit isotonic calibrators.

    #     Three sequential steps:

    #     Step A — L1/L2 blend: for each horizon, grid-search the mixing weight
    #     α ∈ [0, 0.45] between the L1 base (MAE-optimal, clipped target) and
    #     L2 base (MSE-optimal) predictions, minimising validation MAE.

    #     Step B — Spike/dip policy + naive blend: for each horizon that has a
    #     fitted spike chain, select the best spike-correction policy kind
    #     (0=soft blend, 1=gated uplift, 2=hard gate), source (spike_reg /
    #     spike_qreg / max of both), and policy parameters on the spike-aware
    #     composite score. Dip corrections are tuned additively after spike
    #     corrections. Finally, blend the corrected model prediction with a
    #     lag-naive baseline.

    #     Step C — Isotonic calibration: fit a per-horizon isotonic regression
    #     on the fully blended validation predictions to correct residual bias
    #     before inference.

    #     Returns
    #     -------
    #     dict
    #         Keys: blend_l2_alphas, blend_alphas, spike_kind, spike_src,
    #         spike_p1, spike_p2, spike_p3, dip_kind, dip_p1, dip_p2, dip_p3,
    #         calibrators.
    #     """
    #     n_horizons = int(os.environ["HORIZON_HOURS"])

    #     # ── Step A: L1/L2 base blend ──────────────────────────────────────────
    #     blend_l2_alphas = np.zeros(n_horizons, dtype=np.float32)
    #     _l2_alpha_grid  = np.linspace(0.0, 0.45, 10, dtype=np.float32)
    #     for _i, _h in enumerate(horizon_list):
    #         _hf_va  = _compute_target_time_feats(valid_df.index, _h)
    #         _X_va   = np.concatenate([validate_df.values, _hf_va, cross_va], axis=1).astype(np.float32)
    #         _y_tv   = valid_tgt[f"h{_h}"].values.astype(np.float32)
    #         _mask_v = np.isfinite(_y_tv)
    #         if not np.any(_mask_v):
    #             continue
    #         _y_l1 = _from_asinh(base_models[_i].predict(_X_va)).astype(np.float32)
    #         _y_l2 = _from_asinh(base_l2_models[_i].predict(_X_va)).astype(np.float32)
    #         _best_a, _best_mae = 0.0, float("inf")
    #         for _a in _l2_alpha_grid:
    #             _yc  = ((1.0 - _a) * _y_l1 + _a * _y_l2).astype(np.float32)
    #             _mae = float(np.mean(np.abs(_y_tv[_mask_v] - _yc[_mask_v])))
    #             if _mae < _best_mae:
    #                 _best_mae, _best_a = _mae, float(_a)
    #         blend_l2_alphas[_i] = _best_a
    #     print(
    #         f"  L1/L2 blend: mean α={blend_l2_alphas.mean():.3f}  "
    #         f"min={blend_l2_alphas.min():.3f}  max={blend_l2_alphas.max():.3f}",
    #         flush=True,
    #     )

    #     # ── Step B: spike/dip policy + naive blend ────────────────────────────
    #     blend_alphas = np.ones(n_horizons, dtype=np.float32)
    #     alpha_grid   = np.linspace(0.0, 1.0, 41, dtype=np.float32)  # finer grid (was 21)
    #     spike_kind   = np.zeros(n_horizons, dtype=np.int8)     # 0=soft, 1=uplift, 2=hard-gate
    #     spike_src    = np.zeros(n_horizons, dtype=np.int8)     # 0=sr, 1=sq, 2=max(sr,sq)
    #     spike_p1     = np.full(n_horizons, 0.20, dtype=np.float32)
    #     spike_p2     = np.full(n_horizons, 2.0, dtype=np.float32)
    #     spike_p3     = np.full(n_horizons, 0.75, dtype=np.float32)
    #     dip_kind     = np.zeros(n_horizons, dtype=np.int8)     # 0=soft, 1=down-gate, 2=hard-gate
    #     dip_p1       = np.full(n_horizons, 0.15, dtype=np.float32)
    #     dip_p2       = np.full(n_horizons, 1.0, dtype=np.float32)
    #     dip_p3       = np.full(n_horizons, 0.60, dtype=np.float32)

    #     for i, h in enumerate(horizon_list):
    #         naive_col = _horizon_naive_col(h)
    #         if naive_col not in valid_aux.columns:
    #             naive_col = f"{os.environ["TARGET"].lower()}_price_lag_48"
    #         if naive_col not in valid_aux.columns:
    #             continue

    #         naive_h_val = valid_aux[naive_col].values.astype(np.float32)
    #         y_true      = valid_tgt[f"h{h}"].values.astype(np.float32)
    #         mask        = np.isfinite(y_true) & np.isfinite(naive_h_val)
    #         if not np.any(mask):
    #             continue

    #         h_feat_va  = _compute_target_time_feats(valid_df.index, h)
    #         X_h_va = np.concatenate([validate_df.values, h_feat_va, cross_va], axis=1).astype(np.float32)

    #         _l1_val    = _from_asinh(base_models[i].predict(X_h_va)).astype(np.float32)
    #         _l2_val    = _from_asinh(base_l2_models[i].predict(X_h_va)).astype(np.float32)
    #         y_base_val = ((1.0 - blend_l2_alphas[i]) * _l1_val + blend_l2_alphas[i] * _l2_val).astype(np.float32)
    #         y_model    = y_base_val
    #         if spike_clfs[i] is not None:
    #             sp     = spike_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
    #             sr     = _from_asinh(spike_regs[i].predict(X_h_va)).astype(np.float32) if spike_regs[i] else y_base_val
    #             sq     = _from_asinh(spike_qregs[i].predict(X_h_va)).astype(np.float32) if spike_qregs[i] else sr

    #             y_tune = y_true[mask]
    #             b_tune = y_base_val[mask]
    #             s_tune = sp[mask]
    #             r_tune = sr[mask]
    #             q_tune = sq[mask]

    #             # Reference (old behavior): soft blend with spike reg only.
    #             y_ref = _apply_spike_policy(b_tune, s_tune, r_tune, q_tune, 0, 0.0, 0.0, 0.0, 0)
    #             ref_spike = y_tune > SPIKE_THRESHOLD
    #             if int((~ref_spike).sum()) > 0:
    #                 ref_non_mae = float(np.mean(np.abs(y_tune[~ref_spike] - y_ref[~ref_spike])))
    #             else:
    #                 ref_non_mae = float(np.mean(np.abs(y_tune - y_ref)))

    #             best_sc = _spike_policy_score(y_tune, y_ref, SPIKE_THRESHOLD)
    #             best_k, best_src = 0, 0
    #             best_p1, best_p2, best_p3 = 0.0, 0.0, 0.0

    #             def _accept_candidate(y_candidate: np.ndarray) -> tuple[bool, float]:
    #                 m = np.abs(y_tune - y_candidate)
    #                 if int((~ref_spike).sum()) > 0:
    #                     cand_non = float(np.mean(m[~ref_spike]))
    #                 else:
    #                     cand_non = float(np.mean(m))
    #                 if cand_non > ref_non_mae + 0.20:
    #                     return False, float("inf")
    #                 return True, _spike_policy_score(y_tune, y_candidate, SPIKE_THRESHOLD)

    #             # Policy 0: soft blend (src variants)
    #             for src in (0, 1, 2):
    #                 y_try = _apply_spike_policy(b_tune, s_tune, r_tune, q_tune, 0, 0.0, 0.0, 0.0, src)
    #                 ok, sc = _accept_candidate(y_try)
    #                 if ok and sc < best_sc:
    #                     best_sc = sc
    #                     best_k, best_src = 0, src
    #                     best_p1, best_p2, best_p3 = 0.0, 0.0, 0.0

    #             # Policy 1: gated uplift (threshold/power/wmax)
    #             for src in (0, 1, 2):
    #                 for thr in _SPIKE_THR_GRID:
    #                     for pwr in _SPIKE_POW_GRID:
    #                         for wmx in _SPIKE_WMAX_GRID:
    #                             y_try = _apply_spike_policy(
    #                                 b_tune, s_tune, r_tune, q_tune,
    #                                 1, float(thr), float(pwr), float(wmx), src,
    #                             )
    #                             ok, sc = _accept_candidate(y_try)
    #                             if ok and sc < best_sc:
    #                                 best_sc = sc
    #                                 best_k, best_src = 1, src
    #                                 best_p1, best_p2, best_p3 = float(thr), float(pwr), float(wmx)

    #             # Policy 2: hard gate uplift (threshold, weight)
    #             for src in (0, 1, 2):
    #                 for thr in _SPIKE_THR_GRID:
    #                     for gw in _SPIKE_GATE_W_GRID:
    #                         y_try = _apply_spike_policy(
    #                             b_tune, s_tune, r_tune, q_tune,
    #                             2, float(thr), float(gw), 0.0, src,
    #                         )
    #                         ok, sc = _accept_candidate(y_try)
    #                         if ok and sc < best_sc:
    #                             best_sc = sc
    #                             best_k, best_src = 2, src
    #                             best_p1, best_p2, best_p3 = float(thr), float(gw), 0.0

    #             spike_kind[i] = best_k
    #             spike_src[i]  = best_src
    #             spike_p1[i]   = best_p1
    #             spike_p2[i]   = best_p2
    #             spike_p3[i]   = best_p3

    #             y_model = _apply_spike_policy(
    #                 y_base_val,
    #                 sp,
    #                 sr,
    #                 sq,
    #                 int(best_k),
    #                 float(best_p1),
    #                 float(best_p2),
    #                 float(best_p3),
    #                 int(best_src),
    #             )

    #         # Dip policy tuning after spike policy so downside corrections are additive.
    #         if dip_clfs[i] is not None and dip_regs[i] is not None:
    #             dp = dip_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
    #             dr = _from_asinh(dip_regs[i].predict(X_h_va)).astype(np.float32)

    #             y_tune = y_true[mask]
    #             b_tune = y_model[mask]
    #             p_tune = dp[mask]
    #             d_tune = dr[mask]
    #             ref_mid = (y_tune >= _DIP_THRESHOLD) & (y_tune <= SPIKE_THRESHOLD)
    #             if int(ref_mid.sum()) > 0:
    #                 ref_mid_mae = float(np.mean(np.abs(y_tune[ref_mid] - b_tune[ref_mid])))
    #             else:
    #                 ref_mid_mae = float(np.mean(np.abs(y_tune - b_tune)))

    #             best_d_sc = _spike_policy_score(y_tune, b_tune, SPIKE_THRESHOLD)
    #             best_dk = 0
    #             best_dp1, best_dp2, best_dp3 = 0.0, 0.0, 0.0

    #             def _accept_dip(y_candidate: np.ndarray) -> tuple[bool, float]:
    #                 if int(ref_mid.sum()) > 0:
    #                     mid_mae = float(np.mean(np.abs(y_tune[ref_mid] - y_candidate[ref_mid])))
    #                 else:
    #                     mid_mae = float(np.mean(np.abs(y_tune - y_candidate)))
    #                 if mid_mae > ref_mid_mae + 0.15:
    #                     return False, float("inf")
    #                 return True, _spike_policy_score(y_tune, y_candidate, SPIKE_THRESHOLD)

    #             y_try = _apply_dip_policy(b_tune, p_tune, d_tune, 0, 0.0, 0.0, 0.0)
    #             ok, sc = _accept_dip(y_try)
    #             if ok and sc < best_d_sc:
    #                 best_d_sc = sc
    #                 best_dk = 0
    #                 best_dp1, best_dp2, best_dp3 = 0.0, 0.0, 0.0

    #             for thr in _DIP_THR_GRID:
    #                 for pwr in _DIP_POW_GRID:
    #                     for wmx in _DIP_WMAX_GRID:
    #                         y_try = _apply_dip_policy(
    #                             b_tune, p_tune, d_tune, 1, float(thr), float(pwr), float(wmx),
    #                         )
    #                         ok, sc = _accept_dip(y_try)
    #                         if ok and sc < best_d_sc:
    #                             best_d_sc = sc
    #                             best_dk = 1
    #                             best_dp1, best_dp2, best_dp3 = float(thr), float(pwr), float(wmx)

    #             for thr in _DIP_THR_GRID:
    #                 for gw in _DIP_GATE_W_GRID:
    #                     y_try = _apply_dip_policy(
    #                         b_tune, p_tune, d_tune, 2, float(thr), float(gw), 0.0,
    #                     )
    #                     ok, sc = _accept_dip(y_try)
    #                     if ok and sc < best_d_sc:
    #                         best_d_sc = sc
    #                         best_dk = 2
    #                         best_dp1, best_dp2, best_dp3 = float(thr), float(gw), 0.0

    #             dip_kind[i] = best_dk
    #             dip_p1[i]   = best_dp1
    #             dip_p2[i]   = best_dp2
    #             dip_p3[i]   = best_dp3

    #             y_model = _apply_dip_policy(
    #                 y_model, dp, dr,
    #                 int(best_dk), float(best_dp1), float(best_dp2), float(best_dp3),
    #             )

    #         y_t, y_m, y_n = y_true[mask], y_model[mask], naive_h_val[mask]
    #         best_a, best_mae = 1.0, float("inf")
    #         for a in alpha_grid:
    #             mae = float(np.mean(np.abs(y_t - (a * y_m + (1.0 - a) * y_n))))
    #             if mae < best_mae:
    #                 best_mae, best_a = mae, float(a)
    #         blend_alphas[i] = best_a

    #     # ── Step C: Isotonic calibration ──────────────────────────────────────
    #     calibrators: list[IsotonicRegression | None] = [None] * n_horizons

    #     for i, h in enumerate(horizon_list):
    #         naive_col = _horizon_naive_col(h)
    #         if naive_col not in valid_aux.columns:
    #             naive_col = f"{os.environ["TARGET"].lower()}_price_lag_48"
    #         if naive_col not in valid_aux.columns:
    #             continue

    #         h_feat_va  = _compute_target_time_feats(valid_df.index, h)
    #         X_h_va = np.concatenate([validate_df.values, h_feat_va, cross_va], axis=1).astype(np.float32)

    #         _l1_cal    = _from_asinh(base_models[i].predict(X_h_va)).astype(np.float32)
    #         _l2_cal    = _from_asinh(base_l2_models[i].predict(X_h_va)).astype(np.float32)
    #         y_base_val = ((1.0 - blend_l2_alphas[i]) * _l1_cal + blend_l2_alphas[i] * _l2_cal).astype(np.float32)
    #         if spike_clfs[i] is not None:
    #             sp     = spike_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
    #             sr     = _from_asinh(spike_regs[i].predict(X_h_va)).astype(np.float32) if spike_regs[i] else y_base_val
    #             sq     = _from_asinh(spike_qregs[i].predict(X_h_va)).astype(np.float32) if spike_qregs[i] else sr
    #             y_model = _apply_spike_policy(
    #                 y_base_val, sp, sr, sq,
    #                 int(spike_kind[i]), float(spike_p1[i]), float(spike_p2[i]),
    #                 float(spike_p3[i]), int(spike_src[i]),
    #             )
    #         else:
    #             y_model = y_base_val

    #         if i < len(dip_clfs) and dip_clfs[i] is not None and i < len(dip_regs) and dip_regs[i] is not None:
    #             dp = dip_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
    #             dr = _from_asinh(dip_regs[i].predict(X_h_va)).astype(np.float32)
    #             kind = int(dip_kind[i]) if i < len(dip_kind) else 0
    #             p1   = float(dip_p1[i]) if i < len(dip_p1) else 0.15
    #             p2   = float(dip_p2[i]) if i < len(dip_p2) else 1.0
    #             p3   = float(dip_p3[i]) if i < len(dip_p3) else 0.60
    #             y_model = _apply_dip_policy(y_model, dp, dr, kind, p1, p2, p3)

    #         y_naive = valid_aux[naive_col].values.astype(np.float32)
    #         a       = float(blend_alphas[i])
    #         y_blend = a * y_model + (1.0 - a) * y_naive
    #         y_true  = valid_tgt[f"h{h}"].values.astype(np.float32)
    #         mask    = np.isfinite(y_true) & np.isfinite(y_blend)
    #         if int(mask.sum()) < 500:
    #             continue

    #         iso = IsotonicRegression(out_of_bounds="clip")
    #         iso.fit(y_blend[mask], y_true[mask])
    #         calibrators[i] = iso

    #     return dict(
    #         blend_l2_alphas=blend_l2_alphas,
    #         blend_alphas=blend_alphas,
    #         spike_kind=spike_kind,
    #         spike_src=spike_src,
    #         spike_p1=spike_p1,
    #         spike_p2=spike_p2,
    #         spike_p3=spike_p3,
    #         dip_kind=dip_kind,
    #         dip_p1=dip_p1,
    #         dip_p2=dip_p2,
    #         dip_p3=dip_p3,
    #         calibrators=calibrators,
    #     )

    # tuning = _tune_and_calibrate()

    # return {
    #     "base_models":     base_models,
    #     "base_l2_models":  base_l2_models,
    #     "spike_clfs":      spike_clfs,
    #     "spike_regs":      spike_regs,
    #     "spike_qregs":     spike_qregs,
    #     "dip_clfs":        dip_clfs,
    #     "dip_regs":        dip_regs,
    #     "blend_l2_alphas": tuning["blend_l2_alphas"],
    #     "blend_alphas":    tuning["blend_alphas"],
    #     "spike_kind":      tuning["spike_kind"],
    #     "spike_src":       tuning["spike_src"],
    #     "spike_p1":        tuning["spike_p1"],
    #     "spike_p2":        tuning["spike_p2"],
    #     "spike_p3":        tuning["spike_p3"],
    #     "dip_kind":        tuning["dip_kind"],
    #     "dip_p1":          tuning["dip_p1"],
    #     "dip_p2":          tuning["dip_p2"],
    #     "dip_p3":          tuning["dip_p3"],
    #     "calibrators":     tuning["calibrators"],
    #     "clip_thresh":     _clip_thresh,
    # }
